# MedMNIST cross-dataset replication

Replicates the CIFAR-10 gradient-inversion study on **PathMNIST** (colon pathology,
MedMNIST v2) with the model, optimiser, partitioning, seed, round count and train-set
size all held fixed, so the dataset is the only thing that changes.

**Run order — Kaggle wipes the session on close, so do this in one sitting and never
use "Run All".**

| Cell | What | Approx T4 time | Needs |
|------|------|----------------|-------|
| 0 | Library (imports, model, attacks, defenses) | < 1 min | — |
| S | Smoke test — run first, always | ~3 min | 0 |
| M1 | Baseline FedAvg, alpha=0.5, 60 rounds | ~25 min | 0 |
| M2 | Defense x attack matrix, n=10, 95% CI **(headline)** | ~70-110 min | M1 |
| M2b | Gradient-rescaling ablation | ~10 min | M1 |
| M3 | Training-stage axis | ~15 min | M1 |
| M4 | Batch-size axis | ~10 min | M1 |
| M5 | Multi-step axis (optional) | ~30 min | M1 |
| M6 | Privacy-utility retraining (optional) | ~70 min | M1, M2 |
| M7 | Summary + CIFAR side-by-side + LaTeX table | < 1 min | any |

**Before you start:** turn Kaggle *Internet* ON in the Settings panel. `medmnist`
installs from PyPI and pulls the `.npz` from Zenodo. Download the whole output folder
right after M1 and again after M2 — nothing survives a browser close.

In [ ]:
# =============================================================================
# CELL 0 - LIBRARY.  Run this first, every session, before anything else.
# =============================================================================
import os
import shutil, json, math, time, random, subprocess, sys, copy
from dataclasses import dataclass, asdict

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as T

# ---- knobs -------------------------------------------------------------------
SMOKE = False        # True -> tiny everything, for a ~3 minute end-to-end check
FAKE_DATA = False    # True -> synthetic stand-in for PathMNIST (no download needed)
ALLOW_SMOKE_CELLS = False   # normally leave this alone. Cells M1-M6 refuse to run while
                            # SMOKE is True, because forgetting to flip SMOKE back would
                            # otherwise produce real-looking numbers from a 1,200-sample
                            # 2-round run. Set True only for a deliberate fast dry pass.
# ------------------------------------------------------------------------------

try:
    from scipy.optimize import linear_sum_assignment
    from scipy import stats as _scistats
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scipy"])
    from scipy.optimize import linear_sum_assignment
    from scipy import stats as _scistats

from skimage.metrics import peak_signal_noise_ratio as sk_psnr
from skimage.metrics import structural_similarity as sk_ssim

import matplotlib.pyplot as plt
SHOW_FIGS = True     # False -> save to disk only (faster, and safe in a headless run)

if not FAKE_DATA:
    try:
        import medmnist
        from medmnist import INFO
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "medmnist"])
        import medmnist
        from medmnist import INFO

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
HAS_FUNC = hasattr(torch, "func") and hasattr(torch.func, "functional_call")
print("device:", DEVICE, "| torch", torch.__version__, "| torch.func:", HAS_FUNC)
if not HAS_FUNC:
    print("WARNING: torch.func.functional_call is missing (needs torch >= 2.0). "
          "Cells M1-M4 still work; the multi-step cell M5 will not.")


# ----------------------------------------------------------------------------- config
@dataclass
class Config:
    flag: str = "pathmnist"      # MedMNIST v2 subset; RGB, so the CIFAR model is unchanged
    num_clients: int = 10
    alpha: float = 0.5           # 0.5 = non-IID, 100 = near-IID
    rounds: int = 60
    local_epochs: int = 2
    local_bs: int = 64
    lr: float = 0.1
    num_classes: int = 9         # PathMNIST; overwritten from INFO at load time
    in_ch: int = 3
    img: int = 32                # 28 -> 32 so the architecture matches the CIFAR study
    train_cap: int = 50_000      # match CIFAR-10's train-set size exactly
    data_root: str = "/kaggle/working/data"
    out_dir: str = "/kaggle/working/outputs_medmnist"
    seed: int = 42

    # attack budgets. These must match the CIFAR-10 run exactly, otherwise a weaker
    # attacker and a harder dataset become impossible to tell apart.
    # DECIDED: ig_iters = 1200. The thesis text says IG ran for 1,500 iterations
    # (3.design.tex:42, 4.implementation.tex:30), but every notebook we still have
    # defaults ig_attack and ig_attack_batch to iters=1200 with no call site overriding
    # it, and 4.implementation.tex:236 calls 1,200 "a larger budget" - so 1,500 looks
    # like a drafting error in the prose, not a setting that was ever run. Fix the .tex,
    # not this line. Only change this if a Ctrl+F for "iters=" in fydp-v2.ipynb turns up
    # 1500, and if it does, change it BEFORE running M2.
    dlg_iters: int = 300
    ig_iters: int = 1200
    restarts: int = 2

    # sample sizes
    n_victims: int = 10          # headline defense matrix
    n_victims_axis: int = 5      # the realism axes (matches the CIFAR study)


cfg = Config()

if not os.path.isdir("/kaggle/working"):          # running outside Kaggle
    cfg.data_root = os.path.abspath("./data")
    cfg.out_dir = os.path.abspath("./outputs_medmnist")

if SMOKE:
    cfg.num_clients, cfg.rounds, cfg.local_epochs = 4, 2, 1
    cfg.train_cap = 1200
    cfg.dlg_iters, cfg.ig_iters, cfg.restarts = 20, 40, 1
    cfg.n_victims, cfg.n_victims_axis = 2, 2

os.makedirs(cfg.out_dir, exist_ok=True)
os.makedirs(cfg.data_root, exist_ok=True)
RESULTS = {}


def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def save_results(tag=""):
    """Write everything collected so far. Called after every cell, so a crash costs
    one cell rather than the whole session."""
    path = os.path.join(cfg.out_dir, "results_medmnist.json")
    with open(path, "w") as f:
        json.dump({"config": asdict(cfg), "results": RESULTS}, f, indent=2, default=float)
    print(f"[saved] {path} {tag}")


# ----------------------------------------------------------------------------- data
class _Wrapped(Dataset):
    """Index subset of a MedMNIST split, exposing flat integer .labels like torchvision."""
    def __init__(self, base, indices):
        self.base = base
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = np.asarray(base.labels).reshape(-1)[self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        return self.base[int(self.indices[i])]


class _FakeMed(Dataset):
    """Synthetic stand-in with the same interface, for offline pipeline checks."""
    def __init__(self, n, num_classes, img=28, out=32, seed=0):
        g = torch.Generator().manual_seed(seed)
        self.x = torch.rand(n, 3, img, img, generator=g)
        self.labels = np.random.default_rng(seed).integers(0, num_classes, size=(n, 1))
        self.tf = T.Resize((out, out))

    def __len__(self):
        return len(self.x)

    def __getitem__(self, i):
        return self.tf(self.x[i]), int(self.labels[i][0])


def _transform(cfg):
    # Resize first, ToTensor last -> pixels land in [0,1] with no normalisation.
    # The attacks optimise in pixel space, so normalising would silently change the
    # problem and break comparability with the CIFAR-10 run.
    return T.Compose([T.Resize((cfg.img, cfg.img)), T.ToTensor()])


def _to_int(t):
    return int(np.asarray(t).reshape(-1)[0])       # MedMNIST targets arrive as shape (1,)


def load_data(cfg):
    if FAKE_DATA:
        cfg.num_classes, cfg.in_ch = 9, 3
        return (_FakeMed(min(cfg.train_cap, 2000), cfg.num_classes, out=cfg.img, seed=0),
                _FakeMed(400, cfg.num_classes, out=cfg.img, seed=1))

    info = INFO[cfg.flag]
    cls = getattr(medmnist, info["python_class"])
    cfg.num_classes = len(info["label"])
    cfg.in_ch = info["n_channels"]
    kw = dict(transform=_transform(cfg), target_transform=_to_int, download=True)
    try:
        train_full = cls(split="train", root=cfg.data_root, **kw)
        test = cls(split="test", root=cfg.data_root, **kw)
    except TypeError:                              # older medmnist versions have no root=
        train_full = cls(split="train", **kw)
        test = cls(split="test", **kw)

    n = len(train_full)
    rng = np.random.default_rng(cfg.seed)
    keep = np.sort(rng.choice(n, size=min(cfg.train_cap, n), replace=False))
    train = _Wrapped(train_full, keep)
    print(f"{cfg.flag}: train {len(train)} (of {n}), test {len(test)}, "
          f"{cfg.num_classes} classes, {cfg.in_ch}ch, resized to {cfg.img}x{cfg.img}")
    return train, test


def dirichlet_partition(labels, num_clients, alpha, num_classes, seed=42):
    rng = np.random.default_rng(seed)
    labels = np.asarray(labels).reshape(-1)
    client_idx = [[] for _ in range(num_clients)]
    for c in range(num_classes):
        idx_c = np.where(labels == c)[0]
        rng.shuffle(idx_c)
        proportions = rng.dirichlet(alpha * np.ones(num_clients))
        cut = (np.cumsum(proportions) * len(idx_c)).astype(int)[:-1]
        for i, part in enumerate(np.split(idx_c, cut)):
            client_idx[i].extend(part.tolist())
    for i in range(num_clients):
        rng.shuffle(client_idx[i])
    return client_idx


def _labels_of(ds):
    for attr in ("labels", "targets"):
        if hasattr(ds, attr):
            return np.asarray(getattr(ds, attr)).reshape(-1)
    return np.asarray([ds[i][1] for i in range(len(ds))]).reshape(-1)


def make_client_loaders(cfg, train):
    idx = dirichlet_partition(_labels_of(train), cfg.num_clients, cfg.alpha,
                              cfg.num_classes, cfg.seed)
    loaders = [DataLoader(torch.utils.data.Subset(train, idx[i]),
                          batch_size=cfg.local_bs, shuffle=True, num_workers=0)
               for i in range(cfg.num_clients)]
    print("client sizes:", [len(s) for s in idx])
    return loaders, idx


# ----------------------------------------------------------------------------- model
class LeNet(nn.Module):
    """DLG-style 4-weight-layer CNN with Sigmoid activations.

    Sigmoid rather than ReLU is not a style choice. DLG's L-BFGS needs a twice
    differentiable network, which is also why the model stays small. Identical to the
    CIFAR-10 study except the classifier head, which is 9-way here instead of 10-way.
    """
    def __init__(self, num_classes=9, in_ch=3, img=32):
        super().__init__()
        act = nn.Sigmoid
        self.body = nn.Sequential(
            nn.Conv2d(in_ch, 12, 5, padding=2, stride=2), act(),
            nn.Conv2d(12, 12, 5, padding=2, stride=2), act(),
            nn.Conv2d(12, 12, 5, padding=2, stride=1), act(),
        )
        with torch.no_grad():
            flat = self.body(torch.zeros(1, in_ch, img, img)).numel()
        self.fc = nn.Linear(flat, num_classes)

    def forward(self, x):
        x = self.body(x)
        return self.fc(x.view(x.size(0), -1))


def build_model(cfg, device, seed=None):
    """Construct the model. With seed=None the global RNG is left untouched, so building
    a model never consumes randomness that the DP noise or the data shuffling depends on.
    Pass seed= only when the initial weights themselves have to be reproducible."""
    if seed is not None:
        set_seed(seed)
        return LeNet(cfg.num_classes, cfg.in_ch, cfg.img).to(device)
    st = torch.random.get_rng_state()
    m = LeNet(cfg.num_classes, cfg.in_ch, cfg.img).to(device)
    torch.random.set_rng_state(st)
    return m


# ----------------------------------------------------------------------------- FL
def average_weights(states):
    out = copy.deepcopy(states[0])
    for k in out:
        out[k] = torch.stack([s[k].float() for s in states], 0).mean(0)
    return out


@torch.no_grad()
def evaluate(state, loader, cfg, device):
    model = build_model(cfg, device); model.load_state_dict(state); model.eval()
    correct = total = 0
    for x, y in loader:
        pred = model(x.to(device)).argmax(1).cpu()
        correct += (pred == y).sum().item(); total += y.numel()
    return correct / max(total, 1)


def local_train(global_state, loader, cfg, device):
    model = build_model(cfg, device); model.load_state_dict(global_state); model.train()
    opt = torch.optim.SGD(model.parameters(), lr=cfg.lr)
    crit = nn.CrossEntropyLoss()
    for _ in range(cfg.local_epochs):
        for x, y in loader:
            opt.zero_grad()
            crit(model(x.to(device)), y.to(device)).backward()
            opt.step()
    return {k: v.detach().cpu() for k, v in model.state_dict().items()}


def run_federated(cfg, loaders, test_loader, device, snapshot_at=None, tag=""):
    set_seed(cfg.seed)
    init = build_model(cfg, device, seed=cfg.seed)
    gstate = {k: v.detach().cpu() for k, v in init.state_dict().items()}
    snapshot_at = snapshot_at or []
    snaps, acc_hist = {}, []
    if 0 in snapshot_at:
        snaps[0] = copy.deepcopy(gstate)
    t0 = time.time()
    every = max(1, cfg.rounds // 10)
    for r in range(1, cfg.rounds + 1):
        gstate = average_weights([local_train(gstate, ld, cfg, device) for ld in loaders])
        if r % every == 0 or r == cfg.rounds:
            a = evaluate(gstate, test_loader, cfg, device)
            acc_hist.append((r, a))
            print(f"  [{tag}] round {r:3d}/{cfg.rounds}  acc {a:.4f}  ({time.time()-t0:.0f}s)")
        if r in snapshot_at:
            snaps[r] = copy.deepcopy(gstate)
    return gstate, acc_hist, snaps


# ----------------------------------------------------------------------------- defenses
def prune_gradient(grad, ratio):
    if ratio <= 0:
        return [g.clone() for g in grad]
    flat = torch.cat([g.flatten().abs() for g in grad])
    k = max(1, min(int(len(flat) * ratio), len(flat) - 1))
    thresh = torch.kthvalue(flat, k).values
    return [g * (g.abs() > thresh).float() for g in grad]


def dp_gradient(grad, clip_C, sigma):
    if clip_C is not None:
        norm = torch.sqrt(sum((g ** 2).sum() for g in grad))
        scale = min(1.0, float(clip_C) / float(norm + 1e-12))
    else:
        scale = 1.0
    out = []
    for g in grad:
        gg = g * scale
        if sigma and sigma > 0:
            gg = gg + torch.randn_like(gg) * sigma
        out.append(gg)
    return out


def apply_defense(grad, spec):
    g = grad
    if spec.get("prune", 0) > 0:
        g = prune_gradient(g, spec["prune"])
    if spec.get("clip") is not None or spec.get("sigma", 0) > 0:
        g = dp_gradient(g, spec.get("clip"), spec.get("sigma", 0))
    return g


# Noise levels and clip norms are reported directly. No formal (epsilon, delta) claim is
# made anywhere in this study: the per-batch clip-and-noise here is a gradient-perturbation
# heuristic, not a calibrated DP-SGD accountant.
DEFENSES = [
    ("none",         {}),
    ("clip C=4",     {"clip": 4.0}),
    ("noise s=0.01", {"sigma": 0.01}),
    ("noise s=0.1",  {"sigma": 0.1}),
    ("prune 90%",    {"prune": 0.90}),
    ("combined",     {"prune": 0.90, "clip": 4.0, "sigma": 0.01}),
]
if SMOKE:
    DEFENSES = DEFENSES[:3]


def defended_local_train(global_state, loader, cfg, device, spec):
    """Local SGD with the defense applied to every per-batch gradient. Pruning carries an
    error-feedback residual (Deep Gradient Compression style), which is what stops 90%
    pruning from wrecking accuracy."""
    model = build_model(cfg, device); model.load_state_dict(global_state); model.train()
    params = list(model.parameters()); crit = nn.CrossEntropyLoss()
    use_prune = spec.get("prune", 0) > 0
    residual = [torch.zeros_like(p) for p in params] if use_prune else None
    for _ in range(cfg.local_epochs):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y)
            grads = list(torch.autograd.grad(loss, params))
            if use_prune:
                grads = [g + r for g, r in zip(grads, residual)]
                pruned = prune_gradient(grads, spec["prune"])
                residual = [(g - pg).detach() for g, pg in zip(grads, pruned)]
                grads = pruned
            if spec.get("clip") is not None or spec.get("sigma", 0) > 0:
                grads = dp_gradient(grads, spec.get("clip"), spec.get("sigma", 0))
            with torch.no_grad():
                for p, g in zip(params, grads):
                    p -= cfg.lr * g
    return {k: v.detach().cpu() for k, v in model.state_dict().items()}


# ----------------------------------------------------------------------------- gradients
def capture_gradient(state, x, y, cfg, device):
    model = build_model(cfg, device); model.load_state_dict(state); model.eval()
    model.zero_grad()
    loss = nn.CrossEntropyLoss()(model(x.to(device)), y.to(device))
    grads = torch.autograd.grad(loss, list(model.parameters()))
    return [g.detach().cpu().clone() for g in grads]


def multistep_delta(state, x, y, cfg, device, E, lr):
    """Victim side: E local SGD steps on the same batch, transmitted as delta_w / lr,
    so E=1 reduces exactly to the single-step gradient."""
    model = build_model(cfg, device); model.load_state_dict(state); model.eval()
    params = list(model.parameters()); crit = nn.CrossEntropyLoss()
    w0 = [p.detach().clone() for p in params]
    x, y = x.to(device), y.to(device)
    for _ in range(E):
        grads = torch.autograd.grad(crit(model(x), y), params)
        with torch.no_grad():
            for p, g in zip(params, grads):
                p -= lr * g
    return [((a - b.detach()) / lr).cpu().clone() for a, b in zip(w0, params)]


def _simulate_delta(model, names, w0, dx, y, E, lr, crit):
    """Attacker side: the same E steps, unrolled differentiably through dx."""
    w = {n: p for n, p in zip(names, w0)}
    for _ in range(E):
        loss = crit(torch.func.functional_call(model, w, (dx,)), y)
        g = torch.autograd.grad(loss, list(w.values()), create_graph=True)
        w = {n: w[n] - lr * gi for n, gi in zip(names, g)}
    return [(w0i - w[n]) / lr for n, w0i in zip(names, w0)]


# ----------------------------------------------------------------------------- attacks
def total_variation(x):
    dx = (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
    dy = (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
    return dx + dy


def extract_label(grad):
    """iDLG analytic label recovery. Only valid on an undefended single-sample gradient."""
    return int(grad[-1].argmin().item())


def _init_latent(B, cfg, device, seed):
    g = torch.Generator().manual_seed(int(seed))
    z = torch.randn(B, cfg.in_ch, cfg.img, cfg.img, generator=g)
    return z.to(device).requires_grad_(True)


def _prepare(state, cfg, device, real_grad, labels):
    model = build_model(cfg, device); model.load_state_dict(state); model.eval()
    params = list(model.parameters())
    names = [n for n, _ in model.named_parameters()]
    real = [g.detach().to(device) for g in real_grad]
    y = torch.as_tensor(np.atleast_1d(np.asarray(labels)), device=device, dtype=torch.long)
    return model, params, names, real, y


def dlg_attack(state, real_grad, labels, cfg, device, iters=None, lr=1.0,
               restarts=None, E=1, seed_off=0):
    """DLG: minimise the squared L2 distance between simulated and observed gradient.
    Magnitude sensitive, so anything that rescales the gradient hurts it."""
    iters = cfg.dlg_iters if iters is None else iters
    restarts = cfg.restarts if restarts is None else restarts
    model, params, names, real, y = _prepare(state, cfg, device, real_grad, labels)
    B = y.numel(); crit = nn.CrossEntropyLoss()
    if E > 1:
        w0 = [p.detach().clone().requires_grad_(True) for p in params]

    def objective(latent):
        dx = torch.sigmoid(latent)
        if E == 1:
            dg = torch.autograd.grad(crit(model(dx), y), params, create_graph=True)
        else:
            dg = _simulate_delta(model, names, w0, dx, y, E, cfg.lr, crit)
        return sum(((a - b) ** 2).sum() for a, b in zip(dg, real))

    best_loss, best = float("inf"), None
    for r in range(restarts):
        latent = _init_latent(B, cfg, device, cfg.seed + 1000 * seed_off + r)
        opt = torch.optim.LBFGS([latent], lr=lr, max_iter=20, line_search_fn="strong_wolfe")

        def closure():
            opt.zero_grad()
            obj = objective(latent)
            obj.backward()
            return obj

        for _ in range(iters):
            opt.step(closure)
        fl = float(objective(latent).detach())
        if math.isfinite(fl) and fl < best_loss:
            best_loss, best = fl, torch.sigmoid(latent).detach().cpu().clone()

    if best is None:
        best, best_loss = torch.zeros(B, cfg.in_ch, cfg.img, cfg.img), float("nan")
    return best, best_loss


def ig_attack(state, real_grad, labels, cfg, device, iters=None, lr=0.1, tv_weight=1e-2,
              restarts=None, E=1, seed_off=0):
    """Inverting Gradients: minimise cosine distance plus a total-variation prior.
    Direction sensitive and magnitude invariant, which is exactly why clipping does
    not stop it."""
    iters = cfg.ig_iters if iters is None else iters
    restarts = cfg.restarts if restarts is None else restarts
    model, params, names, real, y = _prepare(state, cfg, device, real_grad, labels)
    rn = torch.sqrt(sum((g ** 2).sum() for g in real))
    B = y.numel(); crit = nn.CrossEntropyLoss()
    if E > 1:
        w0 = [p.detach().clone().requires_grad_(True) for p in params]

    def cosine_and_tv(latent):
        dx = torch.sigmoid(latent)
        if E == 1:
            dg = torch.autograd.grad(crit(model(dx), y), params, create_graph=True)
        else:
            dg = _simulate_delta(model, names, w0, dx, y, E, cfg.lr, crit)
        dot = sum((a * b).sum() for a, b in zip(dg, real))
        dn = torch.sqrt(sum((a ** 2).sum() for a in dg))
        cos = 1 - dot / (dn * rn + 1e-8)
        return cos, cos + tv_weight * total_variation(dx)

    best_loss, best = float("inf"), None
    for r in range(restarts):
        latent = _init_latent(B, cfg, device, cfg.seed + 1000 * seed_off + r)
        opt = torch.optim.Adam([latent], lr=lr)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=iters)
        last = float("inf")
        for _ in range(iters):
            opt.zero_grad()
            cos, obj = cosine_and_tv(latent)
            obj.backward()
            opt.step(); sched.step()
            last = float(cos.detach())
        if math.isfinite(last) and last < best_loss:
            best_loss, best = last, torch.sigmoid(latent).detach().cpu().clone()

    if best is None:
        best, best_loss = torch.zeros(B, cfg.in_ch, cfg.img, cfg.img), float("nan")
    return best, best_loss


ATTACKS = {"DLG": dlg_attack, "IG": ig_attack}


# ----------------------------------------------------------------------------- metrics
PSNR_CAP = 100.0   # an exact reconstruction gives MSE 0 and PSNR inf; inf would be
                   # dropped by mean_ci's isfinite filter, so the single most alarming
                   # result would silently vanish from the mean. Cap it instead.


def compute_metrics(recon, real):
    a = recon.permute(1, 2, 0).clamp(0, 1).numpy().astype(np.float64)
    b = real.permute(1, 2, 0).clamp(0, 1).numpy().astype(np.float64)
    mse = float(((a - b) ** 2).mean())
    psnr = PSNR_CAP if mse <= 0 else min(float(sk_psnr(b, a, data_range=1.0)), PSNR_CAP)
    return psnr, float(sk_ssim(b, a, channel_axis=2, data_range=1.0))


def match_and_score(recon, real):
    """Hungarian match on MSE, then mean PSNR/SSIM over the matched pairs. Reduces to a
    plain per-image score when B=1."""
    B = real.size(0)
    cost = np.zeros((B, B))
    for i in range(B):
        for j in range(B):
            cost[i, j] = float(((recon[i] - real[j]) ** 2).mean())
    ri, ci = linear_sum_assignment(cost)
    pairs = [compute_metrics(recon[int(i)], real[int(j)]) for i, j in zip(ri, ci)]
    return (float(np.mean([p for p, _ in pairs])),
            float(np.mean([s for _, s in pairs])),
            [int(j) for j in ci])


def mean_ci(vals, conf=0.95):
    a = np.asarray(vals, dtype=float)
    a = a[np.isfinite(a)]
    n = len(a)
    if n == 0:
        return float("nan"), float("nan")
    m = float(a.mean())
    if n < 2:
        return m, 0.0
    se = float(a.std(ddof=1)) / math.sqrt(n)
    return m, float(se * _scistats.t.ppf(0.5 + conf / 2, n - 1))


def fmt(m, h):
    return f"{m:.3f} +/- {h:.3f}"


def pick_victims(dataset, n, seed):
    """Seeded random draw. MedMNIST test splits are often stored in class order, so
    taking range(n) would sample a single class and quietly bias every number below."""
    rng = np.random.default_rng(int(seed))
    idx = rng.choice(len(dataset), size=min(n, len(dataset)), replace=False)
    return [int(i) for i in np.sort(idx)]


def get_batch(dataset, indices):
    xs, ys = [], []
    for i in indices:
        x, y = dataset[i]
        xs.append(x); ys.append(int(y))
    return torch.stack(xs), torch.tensor(ys, dtype=torch.long)


def _finish(fig, path):
    """Show the figure inline in Kaggle as well as writing the PNG. The reconstruction
    grids are the whole point of looking at this run, so they should not be invisible
    until the output folder is downloaded."""
    print("[fig]", path)
    if SHOW_FIGS:
        plt.show()
    plt.close(fig)


def save_grid(images, titles, path, nrow=6, suptitle=None):
    n = len(images)
    ncol = int(math.ceil(n / nrow))
    fig, axes = plt.subplots(ncol, nrow, figsize=(1.9 * nrow, 2.1 * ncol), squeeze=False)
    for k, ax in enumerate(axes.flat):
        ax.axis("off")
        if k < n:
            ax.imshow(images[k].permute(1, 2, 0).clamp(0, 1).numpy())
            ax.set_title(titles[k], fontsize=8)
    if suptitle:
        fig.suptitle(suptitle, fontsize=11)
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches="tight")
    _finish(fig, path)


def lineplot(xs, series, xlabel, title, path, logx=None, xticks=None):
    fig, ax = plt.subplots(figsize=(5, 3.2))
    for name, (m, e) in series.items():
        ax.errorbar(xs, m, yerr=e, marker="o", capsize=3, label=name)
    if logx:
        ax.set_xscale("log", base=logx)
    if xticks is not None:
        ax.set_xticks(xs); ax.set_xticklabels(xticks)
    ax.set_xlabel(xlabel); ax.set_ylabel("SSIM"); ax.set_title(title)
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
    fig.savefig(path, dpi=160)
    _finish(fig, path)


print("library ready | SMOKE =", SMOKE, "| FAKE_DATA =", FAKE_DATA)

In [ ]:
# =============================================================================
# CELL S - SMOKE TEST.  Set SMOKE = True in Cell 0, re-run Cell 0, run this, then set
# SMOKE back to False and re-run Cell 0. ~3 min. Catches the label-shape, resize,
# partition, attack and multi-step bugs before you spend GPU hours.
# =============================================================================
def smoke():
    t0 = time.time()
    train, test = load_data(cfg)
    loaders, _ = make_client_loaders(cfg, train)
    test_loader = DataLoader(test, batch_size=256)

    m = build_model(cfg, DEVICE)
    print("params:", sum(p.numel() for p in m.parameters()))
    xb, yb = next(iter(loaders[0]))
    assert tuple(xb.shape[1:]) == (cfg.in_ch, cfg.img, cfg.img), xb.shape
    assert yb.dtype == torch.long and yb.dim() == 1, (yb.dtype, yb.shape)
    assert float(xb.min()) >= 0.0 and float(xb.max()) <= 1.0

    gstate, hist, snaps = run_federated(cfg, loaders, test_loader, DEVICE,
                                        snapshot_at=[0, cfg.rounds], tag="smoke")
    assert set(snaps) == {0, cfg.rounds}

    vi = pick_victims(test, 2, cfg.seed)
    x, y = get_batch(test, vi)
    g = capture_gradient(gstate, x[:1], y[:1], cfg, DEVICE)
    print("iDLG label:", extract_label(g), "| true:", int(y[0]))

    for name, fn in ATTACKS.items():
        rec, loss = fn(gstate, g, [int(y[0])], cfg, DEVICE)
        ps, ss, _ = match_and_score(rec, x[:1])
        print(f"  {name} B=1: psnr {ps:.2f} ssim {ss:.3f} loss {loss:.4g}")
        recb, _ = fn(gstate, capture_gradient(gstate, x, y, cfg, DEVICE),
                     y.tolist(), cfg, DEVICE)
        psb, ssb, order = match_and_score(recb, x)
        print(f"  {name} B=2: psnr {psb:.2f} ssim {ssb:.3f} match {order}")

    # the defenses must not change tensor shapes, and the noise must differ per draw
    for nm, spec in DEFENSES:
        gd = apply_defense(g, spec)
        assert all(a.shape == b.shape for a, b in zip(gd, g)), nm
    n1 = apply_defense(g, {"sigma": 0.1}); n2 = apply_defense(g, {"sigma": 0.1})
    assert float((n1[0] - n2[0]).abs().max()) > 0, "DP noise is repeating - RNG is pinned"

    # E=1 must reproduce the single-step gradient exactly
    d1 = multistep_delta(gstate, x[:1], y[:1], cfg, DEVICE, E=1, lr=cfg.lr)
    err = max(float((a - b).abs().max()) for a, b in zip(d1, g))
    print("multi-step E=1 vs single-step max abs err:", err)
    assert err < 1e-4, "multi-step does not reduce to the single-step gradient"

    if HAS_FUNC:
        d2 = multistep_delta(gstate, x[:1], y[:1], cfg, DEVICE, E=2, lr=cfg.lr)
        _ = ig_attack(gstate, d2, [int(y[0])], cfg, DEVICE, E=2, iters=10, restarts=1)
        _ = dlg_attack(gstate, d2, [int(y[0])], cfg, DEVICE, E=2, iters=3, restarts=1)
        print("multi-step attack path ok")

    _ = defended_local_train(gstate, loaders[0], cfg, DEVICE, {"prune": 0.9, "clip": 4.0})
    print(f"SMOKE PASSED in {time.time()-t0:.0f}s")


if SMOKE:
    smoke()

In [ ]:
# =============================================================================
# CELL M1 - BASELINE.  FedAvg on PathMNIST, alpha=0.5, 60 rounds.  ~25 min.
# Download the output folder as soon as this finishes.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."
train, test = load_data(cfg)
test_loader = DataLoader(test, batch_size=256)
loaders, client_idx = make_client_loaders(cfg, train)

mid = cfg.rounds // 2
gstate, acc_hist, snaps = run_federated(cfg, loaders, test_loader, DEVICE,
                                        snapshot_at=[0, mid, cfg.rounds], tag="alpha=0.5")
final_acc = evaluate(gstate, test_loader, cfg, DEVICE)
n_params = sum(p.numel() for p in build_model(cfg, DEVICE).parameters())

# One victim set, drawn once, reused by every later cell so the axes are matched.
victims = pick_victims(test, cfg.n_victims, cfg.seed)
VX, VY = get_batch(test, victims)

RESULTS["baseline"] = {
    "dataset": cfg.flag, "alpha": cfg.alpha, "rounds": cfg.rounds,
    "final_acc": final_acc, "acc_history": acc_hist, "params": n_params,
    "client_sizes": [len(c) for c in client_idx],
    "victims": victims, "victim_labels": VY.tolist(),
}
torch.save({"gstate": gstate, "snaps": snaps, "cfg": asdict(cfg), "victims": victims},
           os.path.join(cfg.out_dir, "med_baseline.pt"))
print(f"\nPathMNIST alpha={cfg.alpha}: final acc {final_acc:.4f} | params {n_params}")
print("victims:", victims, "| labels:", VY.tolist())

plt.figure(figsize=(5, 3.2))
plt.plot([r for r, _ in acc_hist], [a for _, a in acc_hist], marker="o", ms=3)
plt.xlabel("round"); plt.ylabel("test accuracy")
plt.title(f"PathMNIST, {cfg.num_clients} clients, Dirichlet $\\alpha$={cfg.alpha}")
plt.grid(alpha=.3); plt.tight_layout()
_p = os.path.join(cfg.out_dir, "med_acc_alpha0.5.png")
plt.savefig(_p, dpi=160)
_finish(plt.gcf(), _p)
save_results("M1")

In [ ]:
# =============================================================================
# CELL R - RESTORE.  Skips M1 by loading the trained model from disk.
# Looks in the working directory first, then in any attached Kaggle dataset.
# Run this INSTEAD of M1 when resuming after a session drop.
# =============================================================================
def _find(fname):
    cands = [os.path.join(cfg.out_dir, fname)]
    for root in ("/kaggle/input",):
        if os.path.isdir(root):
            for d in os.listdir(root):
                cands.append(os.path.join(root, d, fname))
    for p in cands:
        if os.path.exists(p):
            return p
    return None

os.makedirs(cfg.out_dir, exist_ok=True)
_bp = _find("med_baseline.pt")
assert _bp, "med_baseline.pt not found. Attach it as a Kaggle dataset, or run M1."
print("baseline:", _bp)

ckpt = torch.load(_bp, map_location=DEVICE, weights_only=False)
gstate, snaps, victims = ckpt["gstate"], ckpt["snaps"], ckpt["victims"]

_rp = _find("results_medmnist.json")
if _rp:
    with open(_rp) as f:
        RESULTS = json.load(f)["results"]
    print("results:", _rp, "| keys:", list(RESULTS.keys()))
else:
    print("no prior results json; RESULTS stays empty")

train, test = load_data(cfg)
VX, VY = get_batch(test, victims)
print("victims:", victims, "| labels:", VY.tolist())
print("acc check:", evaluate(gstate, DataLoader(test, batch_size=256), cfg, DEVICE),
      "  (expect ~0.6077)")

# copy anything found in an attached dataset into the working dir so later cells,
# checkpoints and downloads all live in one place
for _f in ("med_baseline.pt", "results_medmnist.json", "med_m2_partial.pt",
           "med_recons.pt", "med_mean_img.pt"):
    _s = _find(_f)
    if _s and not _s.startswith(cfg.out_dir):
        shutil.copy(_s, os.path.join(cfg.out_dir, _f))
        print("copied", _f)


In [ ]:
# =============================================================================
# CELL N - NULL BASELINE.  CPU, under 1 minute.  Run this BEFORE M2.
# What do trivial non-attacks score against these same victims?  On low-variance
# imagery a blurry mean-like image scores well on SSIM/PSNR without containing any
# private information, so an attack that does not clear these numbers has not
# demonstrated reconstruction.  Knowing the floor before M2 runs means the headline
# matrix can be read the moment it prints.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."

_ri = np.random.default_rng(0).choice(len(test), size=min(2000, len(test)), replace=False)
MEAN_IMG = torch.stack([test[int(i)][0] for i in _ri]).mean(0)
_gray = torch.full_like(VX[0], 0.5)

NULLS = {
    "constant 0.5":  lambda k: _gray,
    "dataset mean":  lambda k: MEAN_IMG,
    "uniform noise": lambda k: torch.rand(VX[0].shape,
                                          generator=torch.Generator().manual_seed(1000 + k)),
    "other victim":  lambda k: VX[(k + 5) % len(VX)],
}

null_rows = {}
print(f"{'null baseline':<16}{'PSNR':>18}{'SSIM':>18}")
for _nm, _fn in NULLS.items():
    _ps, _ss = zip(*[compute_metrics(_fn(k), VX[k]) for k in range(len(VX))])
    _pm, _ph = mean_ci(_ps)
    _sm, _sh = mean_ci(_ss)
    null_rows[_nm] = {"psnr_mean": _pm, "psnr_ci": _ph, "ssim_mean": _sm, "ssim_ci": _sh,
                      "psnr_all": list(_ps), "ssim_all": list(_ss)}
    print(f"{_nm:<16}{fmt(_pm, _ph):>18}{fmt(_sm, _sh):>18}")

NULL_SSIM_FLOOR = max(r["ssim_mean"] for r in null_rows.values())
NULL_PSNR_FLOOR = max(r["psnr_mean"] for r in null_rows.values())
print(f"\nfloor an attack must clear:  SSIM {NULL_SSIM_FLOOR:.3f}   PSNR {NULL_PSNR_FLOOR:.3f}")
print("Any attack scoring below this has not demonstrated reconstruction, whatever the\n"
      "absolute number looks like. Relative defense comparisons remain valid either way.")

RESULTS["null_baseline"] = {"n": len(VX), "rows": null_rows,
                            "ssim_floor": NULL_SSIM_FLOOR, "psnr_floor": NULL_PSNR_FLOOR}
save_results("N")
torch.save({"MEAN_IMG": MEAN_IMG}, os.path.join(cfg.out_dir, "med_mean_img.pt"))


In [ ]:
# =============================================================================
# CELL N2 - VICTIM AUDIT.  CPU, seconds.  Per-victim floors and degenerate victims.
# Victim floors span roughly 6x on PathMNIST, so a single scalar floor is the wrong
# instrument. Some tiles (class 1 = background) contain no tissue at all; reconstructing
# them scores well and leaks nothing.
# =============================================================================
try:
    from medmnist import INFO
    _names = INFO[cfg.flag]["label"]
    print("classes:", {int(k): v for k, v in _names.items()})
except Exception as _e:
    print("could not read INFO:", _e)
    _names = {}

_nb = RESULTS["null_baseline"]["rows"]
_srcs = ("constant 0.5", "dataset mean", "other victim")
VICTIM_FLOOR = [max(_nb[s]["ssim_all"][k] for s in _srcs) for k in range(len(VY))]

print(f"\n{'victim':<7}{'class':<30}{'const':>8}{'mean':>8}{'other':>8}{'floor':>8}")
for k in range(len(VY)):
    c = int(VY[k])
    print(f"{k:<7}{str(c) + ' ' + str(_names.get(str(c), '?'))[:26]:<30}"
          f"{_nb['constant 0.5']['ssim_all'][k]:>8.3f}"
          f"{_nb['dataset mean']['ssim_all'][k]:>8.3f}"
          f"{_nb['other victim']['ssim_all'][k]:>8.3f}{VICTIM_FLOOR[k]:>8.3f}")

_deg = [k for k in range(len(VY)) if VICTIM_FLOOR[k] > 0.6]
print(f"\nfloor range {min(VICTIM_FLOOR):.3f} to {max(VICTIM_FLOOR):.3f}  "
      f"(median {float(np.median(VICTIM_FLOOR)):.3f})")
print(f"degenerate victims (floor > 0.6, threshold fixed a priori): {_deg}")
print(f"mean floor all: {float(np.mean(VICTIM_FLOOR)):.3f}   excluding those: "
      f"{float(np.mean([f for k, f in enumerate(VICTIM_FLOOR) if k not in _deg])):.3f}")

RESULTS["null_baseline"]["victim_floor"] = VICTIM_FLOOR
RESULTS["null_baseline"]["degenerate_victims"] = _deg
save_results("N2")

save_grid([VX[k] for k in range(len(VX))],
          [f"v{k} c{int(VY[k])}" for k in range(len(VX))],
          os.path.join(cfg.out_dir, "med_victims.png"),
          nrow=10, suptitle="PathMNIST victims")


In [ ]:
# =============================================================================
# CELL M2R - DEFENSE x ATTACK MATRIX, RESUMABLE.  THE HEADLINE.  ~80 min from cold.
# Identical experiment to M2: same seeds, same order, same numbers.  The only change is
# that it checkpoints after every (defense, attack) pair and skips pairs already done,
# so a session drop costs one condition (~7 min) instead of the entire cell.
# Re-run this cell as many times as needed; it picks up where it stopped.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."

CKPT = os.path.join(cfg.out_dir, "med_m2_partial.pt")
if os.path.exists(CKPT):
    _st = torch.load(CKPT, map_location="cpu", weights_only=False)
    defense_rows = _st["defense_rows"]
    all_recon = {tuple(k.split("|")): list(v) for k, v in _st["all_recon"].items()}
    print(f"resuming: {len(defense_rows)}/{len(DEFENSES) * len(ATTACKS)} done ->",
          sorted(defense_rows.keys()))
else:
    defense_rows, all_recon = {}, {}
    print("starting from scratch")

t0 = time.time()
for di, (dname, spec) in enumerate(DEFENSES):
    for aname, afn in ATTACKS.items():
        key = f"{dname}|{aname}"
        if key in defense_rows:
            continue
        psnrs, ssims, recs = [], [], []
        for k in range(len(VY)):
            x1, y1 = VX[k].unsqueeze(0), VY[k].view(1)
            g = capture_gradient(gstate, x1, y1, cfg, DEVICE)
            set_seed(cfg.seed + 1_000_003 * k + 97 * di)   # depends on (k, di) only,
            g = apply_defense(g, spec)                      # so resuming cannot shift it
            rec, _ = afn(gstate, g, [int(VY[k])], cfg, DEVICE, seed_off=k)
            ps, ss, _ = match_and_score(rec, x1)
            psnrs.append(ps)
            ssims.append(ss)
            recs.append(rec[0].detach().cpu())
        pm, ph = mean_ci(psnrs)
        sm, sh = mean_ci(ssims)
        defense_rows[key] = {"defense": dname, "attack": aname, "n": len(ssims),
                             "psnr_mean": pm, "psnr_ci": ph,
                             "ssim_mean": sm, "ssim_ci": sh,
                             "ssim_all": ssims, "psnr_all": psnrs}
        all_recon[(dname, aname)] = recs

        torch.save({"defense_rows": defense_rows,
                    "all_recon": {f"{d}|{a}": torch.stack(v)
                                  for (d, a), v in all_recon.items()}}, CKPT)
        RESULTS["defense_matrix"] = defense_rows
        save_results(key)
        print(f"{dname:<14} {aname:<4} PSNR {fmt(pm, ph)}  SSIM {fmt(sm, sh)}  "
              f"({time.time() - t0:.0f}s)  [checkpointed]")

print(f"\nall {len(defense_rows)} conditions complete")


In [ ]:
# =============================================================================
# CELL M2P - POST-PROCESS M2.  CPU, seconds.  Figures, paired deltas, floor check.
# Split out from the attack loop so it can be re-run freely without touching the GPU.
# =============================================================================
if "all_recon" not in globals() or not all_recon:
    _st = torch.load(os.path.join(cfg.out_dir, "med_m2_partial.pt"),
                     map_location="cpu", weights_only=False)
    defense_rows = _st["defense_rows"]
    all_recon = {tuple(k.split("|")): list(v) for k, v in _st["all_recon"].items()}

first_recon = {k: v[0] for k, v in all_recon.items()}
torch.save({"all_recon": {f"{d}|{a}": torch.stack(v) for (d, a), v in all_recon.items()},
            "VX": VX, "VY": VY, "victims": victims},
           os.path.join(cfg.out_dir, "med_recons.pt"))

# Paired per-victim differences. The same victims are attacked under every condition, so
# differencing removes the victim-to-victim variance that dominates the unpaired CIs.
paired = {}
print("paired per-victim delta SSIM vs 'none' (negative = defense reduced the attack)")
for aname in ATTACKS:
    base_ss = defense_rows[f"none|{aname}"]["ssim_all"]
    for dname, _ in DEFENSES:
        if dname == "none":
            continue
        dl = [a - b for a, b in zip(defense_rows[f"{dname}|{aname}"]["ssim_all"], base_ss)]
        dm_, dh_ = mean_ci(dl)
        paired[f"{aname}|{dname}"] = {"attack": aname, "defense": dname,
                                      "mean": dm_, "ci": dh_, "all": dl,
                                      "separated": bool(abs(dm_) > dh_)}
        print(f"  {aname:<4} {dname:<14} {fmt(dm_, dh_):>18}   "
              f"separated: {'yes' if abs(dm_) > dh_ else 'no'}")
RESULTS["paired_delta_ssim"] = paired

if "null_baseline" in RESULTS:
    _fl = RESULTS["null_baseline"]["ssim_floor"]
    print(f"\nnull SSIM floor (mean over victims): {_fl:.3f}")
    for aname in ATTACKS:
        _v = defense_rows[f"none|{aname}"]["ssim_mean"]
        print(f"  undefended {aname:<4} {_v:.3f}  "
              f"{'CLEARS' if _v > _fl else 'BELOW'} the floor")
    print("  (Cell A does this per victim, which is the version that counts.)")

RESULTS["defense_matrix"] = defense_rows
save_results("M2P")

save_grid([VX[0]] + [first_recon[(d, a)] for d, _ in DEFENSES for a in ATTACKS],
          ["original"] + [f"{d}\n{a}" for d, _ in DEFENSES for a in ATTACKS],
          os.path.join(cfg.out_dir, "med_grid_defense.png"), nrow=7,
          suptitle="PathMNIST reconstructions under each defense")

names_ = [d for d, _ in DEFENSES]
fig, ax = plt.subplots(figsize=(7, 3.4))
w = 0.38
for i, aname in enumerate(ATTACKS):
    m = [defense_rows[f"{d}|{aname}"]["ssim_mean"] for d in names_]
    e = [defense_rows[f"{d}|{aname}"]["ssim_ci"] for d in names_]
    ax.bar(np.arange(len(names_)) + (i - 0.5) * w, m, w, yerr=e, capsize=3, label=aname)
ax.set_xticks(range(len(names_)))
ax.set_xticklabels(names_, rotation=20, ha="right")
ax.set_ylabel("SSIM")
ax.set_title(f"PathMNIST: defense x attack (n={cfg.n_victims}, 95% CI)")
if "null_baseline" in RESULTS:
    ax.axhline(RESULTS["null_baseline"]["ssim_floor"], ls="--", lw=1, color="k")
    ax.text(len(names_) - 0.5, RESULTS["null_baseline"]["ssim_floor"],
            " null floor", va="bottom", ha="right", fontsize=8)
ax.legend()
ax.grid(axis="y", alpha=.3)
fig.tight_layout()
_p = os.path.join(cfg.out_dir, "med_defense_x_attack.png")
fig.savefig(_p, dpi=160)
_finish(fig, _p)


In [ ]:
# =============================================================================
# CELL I - IDENTIFIABILITY.  Pure numpy on reconstructions M2 already produced.  Seconds.
# Absolute SSIM is floor-bound on low-variance imagery (see Cell N), so the question that
# actually discriminates leakage from no leakage is relative: does a reconstruction
# resemble ITS OWN victim more than it resembles the other nine?
#   mean rank  - chance is (n+1)/2; 1.0 means perfect identification
#   top-1      - chance is n * (1/n) = 1
#   same-class - controls for the label being handed to the attack.  If the signal is
#                class-typical texture rather than the individual image, this sits at
#                chance (half the pairs) even when the overall rank looks good.
# =============================================================================
if "all_recon" not in globals():
    _d = torch.load(os.path.join(cfg.out_dir, "med_recons.pt"),
                    map_location="cpu", weights_only=False)
    all_recon = {tuple(k.split("|")): list(v) for k, v in _d["all_recon"].items()}

n_id = len(VY)
lab = VY.tolist()
same_pairs = [(i, j) for i in range(n_id) for j in range(n_id)
              if i != j and lab[i] == lab[j]]

print(f"chance:  mean rank {(n_id + 1) / 2:.2f}   top-1 1/{n_id}   "
      f"same-class {len(same_pairs) / 2:.0f}/{len(same_pairs)}\n")
print(f"{'condition':<20}{'matched':>16}{'mismatched':>16}{'gap':>16}"
      f"{'rank':>8}{'top1':>8}{'sameclass':>11}")

S_ALL, ident_rows = {}, {}
for (_dn, _an), _recs in all_recon.items():
    S = np.array([[compute_metrics(_recs[i], VX[j])[1] for j in range(n_id)]
                  for i in range(n_id)])
    key = f"{_dn}|{_an}"
    S_ALL[key] = S
    matched = [float(S[i, i]) for i in range(n_id)]
    mismatch = [float(S[i, j]) for i in range(n_id) for j in range(n_id) if i != j]
    gap = [float(S[i, i] - np.delete(S[i], i).mean()) for i in range(n_id)]
    ranks = [1 + int((S[i] > S[i, i]).sum()) for i in range(n_id)]
    top1 = int(sum(r == 1 for r in ranks))
    scw = int(sum(bool(S[i, i] > S[i, j]) for i, j in same_pairs))
    mm, mh = mean_ci(matched)
    xm, xh = mean_ci(mismatch)
    gm, gh = mean_ci(gap)
    ident_rows[key] = {"defense": _dn, "attack": _an, "n": n_id,
                       "matched": [mm, mh], "mismatched": [xm, xh], "gap": [gm, gh],
                       "gap_all": gap, "ranks": ranks,
                       "mean_rank": float(np.mean(ranks)), "top1": top1,
                       "same_class_wins": scw, "same_class_n": len(same_pairs),
                       "separated": bool(abs(gm) > gh)}
    print(f"{_dn + ' ' + _an:<20}{fmt(mm, mh):>16}{fmt(xm, xh):>16}{fmt(gm, gh):>16}"
          f"{np.mean(ranks):>8.2f}{str(top1) + '/' + str(n_id):>8}"
          f"{str(scw) + '/' + str(len(same_pairs)):>11}")

_k = "none|IG"
if _k in S_ALL:
    print(f"\nsame-class head-to-head detail, {_k} "
          f"(own victim vs the other victim of the same class):")
    for i, j in same_pairs:
        print(f"  recon {i} (class {lab[i]}): own {S_ALL[_k][i, i]:.3f}  "
              f"vs victim {j} {S_ALL[_k][i, j]:.3f}  "
              f"{'OWN' if S_ALL[_k][i, i] > S_ALL[_k][i, j] else 'other'}")

RESULTS["identifiability"] = ident_rows
save_results("I")
np.savez(os.path.join(cfg.out_dir, "med_cross_ssim.npz"),
         **{k.replace("|", "__"): v for k, v in S_ALL.items()})
print("\n[saved]", os.path.join(cfg.out_dir, "med_cross_ssim.npz"))


In [ ]:
# =============================================================================
# CELL A - FLOOR-RELATIVE ANALYSIS.  CPU, seconds.  Run after Cell I.
# Per-victim floors span ~6x, so comparing mean attack SSIM to a mean floor is not
# meaningful. This scores each attack against EACH victim's own floor, paired, and
# reports how many individual victims the attack actually beat.
# =============================================================================
FL = RESULTS["null_baseline"]["victim_floor"]
DEG = set(RESULTS["null_baseline"].get("degenerate_victims", []))
keep = [k for k in range(len(VY)) if k not in DEG]


def _excess(ss, idx):
    ex = [ss[k] - FL[k] for k in idx]
    m, h = mean_ci(ex)
    return m, h, int(sum(1 for e in ex if e > 0)), len(idx)


print(f"{'condition':<20}{'SSIM':>9}{'excess vs floor':>20}{'beat':>7}"
      f"{'excess (no bg)':>20}{'beat':>7}")
floor_rows = {}
for key, row in RESULTS["defense_matrix"].items():
    ss = row["ssim_all"]
    a_m, a_h, a_w, a_n = _excess(ss, list(range(len(VY))))
    k_m, k_h, k_w, k_n = _excess(ss, keep)
    floor_rows[key] = {"excess_all": [a_m, a_h], "beat_all": a_w, "n_all": a_n,
                       "excess_keep": [k_m, k_h], "beat_keep": k_w, "n_keep": k_n,
                       "ssim_keep_mean": float(np.mean([ss[k] for k in keep]))}
    print(f"{key:<20}{row['ssim_mean']:>9.3f}{fmt(a_m, a_h):>20}"
          f"{f'{a_w}/{a_n}':>7}{fmt(k_m, k_h):>20}{f'{k_w}/{k_n}':>7}")

RESULTS["floor_relative"] = floor_rows
print(f"\nexcluded as background tiles: {sorted(DEG)}")
print("'beat' = victims where the attack exceeded that victim's own trivial floor.")
save_results("A")


In [ ]:
# =============================================================================
# CELL M2b - GRADIENT-RESCALING ABLATION.  ~10 min.
# Multiply the transmitted gradient by k and change nothing else. If the spine of the
# paper holds, IG is flat in k and DLG collapses away from k=1. This turns the
# magnitude-versus-direction claim from an interpretation into a measurement.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."
SCALES = [0.1, 0.5, 1.0, 2.0, 10.0] if not SMOKE else [0.5, 1.0]
n_rs = min(5, len(VY))
rescale_rows = {}
t0 = time.time()
for s in SCALES:
    for aname, afn in ATTACKS.items():
        ssims, losses = [], []
        for k in range(n_rs):
            x1, y1 = VX[k].unsqueeze(0), VY[k].view(1)
            g = [gg * s for gg in capture_gradient(gstate, x1, y1, cfg, DEVICE)]
            rec, fl = afn(gstate, g, [int(VY[k])], cfg, DEVICE, seed_off=k)
            _, ss, _ = match_and_score(rec, x1)
            ssims.append(ss); losses.append(fl)
        m, h = mean_ci(ssims)
        lm, _ = mean_ci(losses)
        rescale_rows[f"{s}|{aname}"] = {"scale": s, "attack": aname, "n": n_rs,
                                        "ssim_mean": m, "ssim_ci": h, "ssim_all": ssims,
                                        "obj_mean": lm, "obj_all": losses}
        print(f"scale x{s:<5} {aname:<4} SSIM {fmt(m, h)}   final objective {lm:.4g}"
              f"  ({time.time()-t0:.0f}s)")

# The objective column is the mechanistic half of the claim: IG's cosine distance should
# barely move with k, while DLG's squared-l2 should scale roughly as k^2. SSIM is the
# downstream consequence; the objective is the cause.
print("\nfinal objective relative to k=1 (the actual test of magnitude invariance)")
for aname in ATTACKS:
    base = rescale_rows[f"1.0|{aname}"]["obj_mean"]
    ratios = [rescale_rows[f"{s}|{aname}"]["obj_mean"] / base
              if base not in (0, None) and math.isfinite(base) else float("nan")
              for s in SCALES]
    print(f"  {aname:<4} " + "  ".join(f"k={s}: {r:8.3g}" for s, r in zip(SCALES, ratios)))
RESULTS["gradient_rescaling"] = rescale_rows
save_results("M2b")
lineplot(SCALES,
         {a: ([rescale_rows[f"{s}|{a}"]["ssim_mean"] for s in SCALES],
              [rescale_rows[f"{s}|{a}"]["ssim_ci"] for s in SCALES]) for a in ATTACKS},
         "gradient rescaling factor $k$", "Magnitude invariance of IG",
         os.path.join(cfg.out_dir, "med_rescaling.png"), logx=10)

In [ ]:
# =============================================================================
# CELL M7 - SUMMARY.  Prints the CIFAR-10 side-by-side and writes a LaTeX table.
# Safe to run after any subset of the cells above; it skips whatever is missing.
# =============================================================================
CIFAR = {   # locked CIFAR-10 numbers from the thesis (mean over 5 victims, no CIs)
    "baseline_acc": 0.460,
    "stage": {"untrained": {"DLG": 0.345, "IG": 0.093},
              "mid":       {"DLG": 0.657, "IG": 0.331},
              "converged": {"DLG": 0.768, "IG": 0.485}},
    "batch": {1: {"DLG": 0.747, "IG": 0.514}, 16: {"DLG": 0.114, "IG": 0.183}},
    "multistep": {1: {"DLG": 0.787, "IG": 0.48}, 8: {"DLG": 0.434, "IG": 0.48}},
    "defense": {"none":         {"DLG": 0.784, "IG": 0.476},
                "clip C=4":     {"DLG": 0.160, "IG": 0.498},
                "noise s=0.01": {"DLG": 0.146, "IG": 0.461},
                "noise s=0.1":  {"DLG": None,  "IG": 0.161},
                "prune 90%":    {"DLG": None,  "IG": 0.252},
                "combined":     {"DLG": None,  "IG": 0.243}},
    "pu": {"none": 0.461, "noise s=0.1": 0.437, "prune 90%": 0.464, "combined": 0.462},
}


def _c(v):
    if v is None or (isinstance(v, float) and not math.isfinite(v)):
        return "n/a"
    return f"{v:.3f}"


def _cell(row):
    if not row:
        return "n/a"
    return fmt(row.get("ssim_mean", float("nan")), row.get("ssim_ci", float("nan")))


print("\n" + "=" * 78)
print("PathMNIST vs CIFAR-10   (SSIM; PathMNIST shown as mean +/- 95% CI)")
print("=" * 78)

if "baseline" in RESULTS:
    print(f"\nbaseline accuracy    PathMNIST {RESULTS['baseline']['final_acc']:.3f}"
          f"   |   CIFAR-10 {CIFAR['baseline_acc']:.3f}")

if "defense_matrix" in RESULTS:
    dm = RESULTS["defense_matrix"]
    print("\nDEFENSE x ATTACK")
    print(f"{'defense':<14}{'DLG path':<20}{'DLG cifar':<11}{'IG path':<20}{'IG cifar':<11}")
    for d, _ in DEFENSES:
        cd = CIFAR["defense"].get(d, {})
        print(f"{d:<14}{_cell(dm.get(d + '|DLG')):<20}{_c(cd.get('DLG')):<11}"
              f"{_cell(dm.get(d + '|IG')):<20}{_c(cd.get('IG')):<11}")
    nd = dm.get("none|IG", {}).get("ssim_mean", float("nan"))
    cl = dm.get("clip C=4|IG", {}).get("ssim_mean", float("nan"))
    if math.isfinite(nd) and math.isfinite(cl):
        print(f"\n  clip-bypass check: IG undefended {nd:.3f} vs IG clipped {cl:.3f} "
              f"({100 * cl / max(nd, 1e-9):.0f}% retained)")

if "gradient_rescaling" in RESULTS:
    rr = RESULTS["gradient_rescaling"]
    print("\nGRADIENT RESCALING (direct test of magnitude invariance)")
    for s in SCALES:
        print(f"  k={s:<6} DLG {_cell(rr.get(f'{s}|DLG')):<20} IG {_cell(rr.get(f'{s}|IG'))}")

if "training_stage" in RESULTS:
    ts = RESULTS["training_stage"]
    print("\nTRAINING STAGE")
    for s, ck in zip(stages, ["untrained", "mid", "converged"]):
        for a in ATTACKS:
            print(f"  {ck:<10} {a:<4} path {_cell(ts.get(f'{s}|{a}')):<20}"
                  f" cifar {_c(CIFAR['stage'][ck][a])}")

if "batch_size" in RESULTS:
    bs = RESULTS["batch_size"]
    print("\nBATCH SIZE")
    for B in BATCHES:
        for a in ATTACKS:
            print(f"  B={B:<3} {a:<4} path {_cell(bs.get(f'{B}|{a}')):<20}"
                  f" cifar {_c(CIFAR['batch'].get(B, {}).get(a))}")

if "multistep" in RESULTS:
    ms = RESULTS["multistep"]
    print("\nMULTI-STEP")
    for E in ESTEPS:
        for a in ATTACKS:
            print(f"  E={E:<3} {a:<4} path {_cell(ms.get(f'{E}|{a}')):<20}"
                  f" cifar {_c(CIFAR['multistep'].get(E, {}).get(a))}")

if "privacy_utility" in RESULTS:
    print("\nPRIVACY-UTILITY")
    for d, row in RESULTS["privacy_utility"].items():
        print(f"  {d:<14} acc path {row['acc']:.3f}  cifar {_c(CIFAR['pu'].get(d))}"
              f"   IG SSIM {_c(row['ig_ssim'])}")


if "null_baseline" in RESULTS:
    nbr = RESULTS["null_baseline"]
    print("\nNULL BASELINE (trivial non-attacks on the same victims)")
    for k, r in nbr["rows"].items():
        print(f"  {k:<16} PSNR {fmt(r['psnr_mean'], r['psnr_ci']):<18}"
              f" SSIM {fmt(r['ssim_mean'], r['ssim_ci'])}")
    print(f"  floor: SSIM {nbr['ssim_floor']:.3f}  PSNR {nbr['psnr_floor']:.3f}")
    if "defense_matrix" in RESULTS:
        for a in ATTACKS:
            v = RESULTS["defense_matrix"].get(f"none|{a}", {}).get("ssim_mean")
            if v is not None:
                print(f"  undefended {a:<4} {v:.3f} -> "
                      f"{'CLEARS' if v > nbr['ssim_floor'] else 'BELOW'} floor")

if "paired_delta_ssim" in RESULTS:
    print("\nPAIRED DELTA SSIM vs no defense")
    for k, r in RESULTS["paired_delta_ssim"].items():
        print(f"  {k:<22} {fmt(r['mean'], r['ci']):>18}   "
              f"separated: {'yes' if r['separated'] else 'no'}")

if "identifiability" in RESULTS:
    ir = RESULTS["identifiability"]
    n_ = next(iter(ir.values()))["n"]
    print(f"\nIDENTIFIABILITY  (chance: rank {(n_ + 1) / 2:.2f}, top-1 1/{n_})")
    print(f"  {'condition':<20}{'gap':>18}{'rank':>8}{'top1':>8}{'sameclass':>11}")
    for k, r in ir.items():
        print(f"  {k:<20}{fmt(r['gap'][0], r['gap'][1]):>18}{r['mean_rank']:>8.2f}"
              f"{str(r['top1']) + '/' + str(r['n']):>8}"
              f"{str(r['same_class_wins']) + '/' + str(r['same_class_n']):>11}")


# ---- LaTeX table for the thesis ---------------------------------------------
def _tex_name(d):
    return (d.replace("%", r"\%")
             .replace("s=", r"$\sigma$=")
             .replace("C=", r"$C$="))


def _tex_pm(row):
    if not row or not math.isfinite(row.get("ssim_mean", float("nan"))):
        return "--"
    return "$" + f"{row['ssim_mean']:.3f}" + r" \pm " + f"{row['ssim_ci']:.3f}" + "$"


def _tex_plain(v):
    return "--" if v is None else f"{v:.3f}"


def latex_defense_table():
    if "defense_matrix" not in RESULTS:
        return ""
    dm = RESULTS["defense_matrix"]
    head = [
        r"\begin{table}[H]",
        r"\centering",
        r"\caption{Reconstruction quality (SSIM) under each defense on PathMNIST, with the "
        r"CIFAR-10 study shown for reference. PathMNIST values are means over $n=" +
        str(cfg.n_victims) + r"$ victims with 95\% confidence intervals; CIFAR-10 values are "
        r"means over 5 victims reported without intervals.}",
        r"\label{tab:med_defense}",
        r"\begin{tabular}{|l|c|c|c|c|}",
        r"\hline",
        r"\textbf{Defense} & \textbf{DLG (Path)} & \textbf{DLG (CIFAR)} & "
        r"\textbf{IG (Path)} & \textbf{IG (CIFAR)} \\",
        r"\hline",
    ]
    body = []
    for d, _ in DEFENSES:
        cd = CIFAR["defense"].get(d, {})
        body.append(" & ".join([_tex_name(d),
                                _tex_pm(dm.get(d + "|DLG")),
                                _tex_plain(cd.get("DLG")),
                                _tex_pm(dm.get(d + "|IG")),
                                _tex_plain(cd.get("IG"))]) + r" \\")
        body.append(r"\hline")
    return "\n".join(head + body + [r"\end{tabular}", r"\end{table}"])


tex = latex_defense_table()
if tex:
    p = os.path.join(cfg.out_dir, "med_defense_table.tex")
    with open(p, "w") as f:
        f.write(tex + "\n")
    print("\n[tex]", p, "\n")
    print(tex)

save_results("M7")
print("\nEverything is in", cfg.out_dir, "- download it before closing the session.")

---

## Optional cells below this line

M3-M6 are **not part of the planned run**. Axis sweeps over an attack that does not
clear the null baseline measure the degradation of noise into noise. Run them only if
Cell I shows genuine individual leakage (mean rank well under chance **and** same-class
head-to-head well above half). M6 is the exception: it measures accuracy under each
defense, which is a utility result that does not depend on any attack succeeding.


In [ ]:
# =============================================================================
# CELL M3 - TRAINING-STAGE AXIS.  untrained / mid / converged.  ~15 min.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."
SX, SY = VX[:cfg.n_victims_axis], VY[:cfg.n_victims_axis]
stages = ["untrained", f"mid (r{mid})", f"converged (r{cfg.rounds})"]
stage_states = dict(zip(stages, [snaps[0], snaps[mid], snaps[cfg.rounds]]))
stage_rows, stage_imgs = {}, {}
t0 = time.time()
for rname in stages:
    st = stage_states[rname]
    for aname, afn in ATTACKS.items():
        ssims, psnrs = [], []
        for k in range(len(SY)):
            x1, y1 = SX[k].unsqueeze(0), SY[k].view(1)
            g = capture_gradient(st, x1, y1, cfg, DEVICE)
            rec, _ = afn(st, g, [int(SY[k])], cfg, DEVICE, seed_off=k)
            ps, ss, _ = match_and_score(rec, x1)
            ssims.append(ss); psnrs.append(ps)
            if k == 0:
                stage_imgs[(rname, aname)] = rec[0]
        m, h = mean_ci(ssims); pm, ph = mean_ci(psnrs)
        stage_rows[f"{rname}|{aname}"] = {"stage": rname, "attack": aname, "n": len(ssims),
                                          "ssim_mean": m, "ssim_ci": h, "ssim_all": ssims,
                                          "psnr_mean": pm, "psnr_ci": ph}
        print(f"{rname:<20} {aname:<4} SSIM {fmt(m, h)}  ({time.time()-t0:.0f}s)")

RESULTS["training_stage"] = stage_rows
save_results("M3")
lineplot([0, 1, 2],
         {a: ([stage_rows[f"{s}|{a}"]["ssim_mean"] for s in stages],
              [stage_rows[f"{s}|{a}"]["ssim_ci"] for s in stages]) for a in ATTACKS},
         "training stage", "PathMNIST: reconstruction vs training stage",
         os.path.join(cfg.out_dir, "med_axis_training_stage.png"),
         xticks=["untrained", "mid", "converged"])
save_grid([SX[0]] + [stage_imgs[(s, a)] for s in stages for a in ATTACKS],
          ["original"] + [f"{s.split(' ')[0]}\n{a}" for s in stages for a in ATTACKS],
          os.path.join(cfg.out_dir, "med_grid_training_stage.png"), nrow=7)

In [ ]:
# =============================================================================
# CELL M4 - BATCH-SIZE AXIS.  B in {1,4,8,16}, Hungarian matching.  ~10 min.
# Each repeat draws a fresh batch, so the error bars reflect batch composition and not
# just optimiser noise.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."
BATCHES = [1, 4, 8, 16] if not SMOKE else [1, 2]
batch_rows = {}
t0 = time.time()
for B in BATCHES:
    for aname, afn in ATTACKS.items():
        ssims = []
        for rep in range(cfg.n_victims_axis):
            idx = pick_victims(test, B, cfg.seed + 7919 * rep + 31 * B)
            xb, yb = get_batch(test, idx)
            g = capture_gradient(gstate, xb, yb, cfg, DEVICE)
            rec, _ = afn(gstate, g, yb.tolist(), cfg, DEVICE, seed_off=rep)
            _, ss, _ = match_and_score(rec, xb)
            ssims.append(ss)
        m, h = mean_ci(ssims)
        batch_rows[f"{B}|{aname}"] = {"batch": B, "attack": aname, "n": len(ssims),
                                      "ssim_mean": m, "ssim_ci": h, "ssim_all": ssims}
        print(f"B={B:<3} {aname:<4} SSIM {fmt(m, h)}  ({time.time()-t0:.0f}s)")

RESULTS["batch_size"] = batch_rows
save_results("M4")
lineplot(BATCHES,
         {a: ([batch_rows[f"{B}|{a}"]["ssim_mean"] for B in BATCHES],
              [batch_rows[f"{B}|{a}"]["ssim_ci"] for B in BATCHES]) for a in ATTACKS},
         "batch size", "PathMNIST: reconstruction vs batch size",
         os.path.join(cfg.out_dir, "med_axis_batch_size.png"), logx=2, xticks=BATCHES)

In [ ]:
# =============================================================================
# CELL M5 - MULTI-STEP AXIS (optional).  E in {1,2,4,8}.  ~30 min.  Needs torch >= 2.0.
# E=1 must land on the M2 "none" numbers. If it does not, stop and debug rather than
# reporting the curve.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."
assert HAS_FUNC, "multi-step needs torch.func.functional_call (torch >= 2.0)"
ESTEPS = [1, 2, 4, 8] if not SMOKE else [1, 2]
ms_rows = {}
t0 = time.time()
for E in ESTEPS:
    for aname, afn in ATTACKS.items():
        ssims = []
        for k in range(cfg.n_victims_axis):
            x1, y1 = VX[k].unsqueeze(0), VY[k].view(1)
            d = multistep_delta(gstate, x1, y1, cfg, DEVICE, E, cfg.lr)
            rec, _ = afn(gstate, d, [int(VY[k])], cfg, DEVICE, E=E, restarts=1, seed_off=k)
            _, ss, _ = match_and_score(rec, x1)
            ssims.append(ss)
        m, h = mean_ci(ssims)
        ms_rows[f"{E}|{aname}"] = {"E": E, "attack": aname, "n": len(ssims),
                                   "ssim_mean": m, "ssim_ci": h, "ssim_all": ssims}
        print(f"E={E:<3} {aname:<4} SSIM {fmt(m, h)}  ({time.time()-t0:.0f}s)")

RESULTS["multistep"] = ms_rows
save_results("M5")
lineplot(ESTEPS,
         {a: ([ms_rows[f"{E}|{a}"]["ssim_mean"] for E in ESTEPS],
              [ms_rows[f"{E}|{a}"]["ssim_ci"] for E in ESTEPS]) for a in ATTACKS},
         "local steps $E$", "PathMNIST: reconstruction vs local steps",
         os.path.join(cfg.out_dir, "med_axis_multistep.png"))

In [ ]:
# =============================================================================
# CELL M6 - PRIVACY-UTILITY (optional).  Retrains under each defense.  ~70 min.
# Every point, including "none", goes through defended_local_train, so the training loop
# is identical across configurations and only the defense differs.
# =============================================================================
assert not SMOKE or ALLOW_SMOKE_CELLS, \
    "SMOKE is still True. Set SMOKE = False in Cell 0 and re-run Cell 0 before this cell."
PU = [("none", {}), ("noise s=0.1", {"sigma": 0.1}),
      ("prune 90%", {"prune": 0.90}),
      ("combined", {"prune": 0.90, "clip": 4.0, "sigma": 0.01})]
if SMOKE:
    PU = PU[:2]

pu_rows = {}
for dname, spec in PU:
    set_seed(cfg.seed)
    st = {k: v.detach().cpu() for k, v in
          build_model(cfg, DEVICE, seed=cfg.seed).state_dict().items()}
    set_seed(cfg.seed)
    t0 = time.time()
    for r in range(1, cfg.rounds + 1):
        st = average_weights([defended_local_train(st, ld, cfg, DEVICE, spec)
                              for ld in loaders])
        if r % max(1, cfg.rounds // 5) == 0:
            print(f"  [{dname}] round {r}/{cfg.rounds} "
                  f"acc {evaluate(st, test_loader, cfg, DEVICE):.4f} ({time.time()-t0:.0f}s)")
    acc = evaluate(st, test_loader, cfg, DEVICE)
    priv = RESULTS.get("defense_matrix", {}).get(f"{dname}|IG", {}).get("ssim_mean", float("nan"))
    pu_rows[dname] = {"defense": dname, "acc": acc, "ig_ssim": priv}
    print(f"{dname:<14} acc {acc:.4f}  IG SSIM {priv:.3f}")

RESULTS["privacy_utility"] = pu_rows
save_results("M6")

fig, ax = plt.subplots(figsize=(5, 3.4))
for dname, row in pu_rows.items():
    ax.scatter(row["ig_ssim"], row["acc"], s=60)
    ax.annotate(dname, (row["ig_ssim"], row["acc"]), fontsize=8,
                xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("IG reconstruction SSIM (lower is more private)")
ax.set_ylabel("test accuracy"); ax.set_title("PathMNIST: privacy-utility")
ax.grid(alpha=.3); fig.tight_layout()
_p = os.path.join(cfg.out_dir, "med_privacy_utility.png")
fig.savefig(_p, dpi=160)
_finish(fig, _p)